# 02 — Feature Engineering

## Objectif

Ce notebook prépare les variables explicatives issues du dataset nettoyé
`data_2024_cleaned.csv` en vue de la modélisation des émissions de CO₂ WLTP.

Les objectifs sont :

- analyser la pertinence des variables disponibles après nettoyage ;
- identifier les variables présentant un risque de fuite de données ;
- construire les variables dérivées utiles à la modélisation ;
- analyser la cardinalité des variables catégorielles ;
- définir la stratégie d'encodage à appliquer ultérieurement dans le pipeline ML ;
- produire un dataset de features non encodées, reproductible et exploitable
  pour la séparation train / validation / test.

La variable cible de régression est :

`co2_wltp_g_km`

Les encodeurs statistiques ou catégoriels appris sur les données ne sont pas
ajustés dans ce notebook afin d'éviter toute fuite de données avant la séparation
des jeux d'entraînement et de test.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Configuration du projet
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "02_feature_engineering":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

INPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "data_2024_cleaned.csv"
)

TARGET = "co2_wltp_g_km"

TEST_MODE = True
NROWS_TEST = 100_000

## 1. Chargement du dataset nettoyé

### Objectif

Cette étape charge le dataset produit par le pipeline de nettoyage.

Deux modes sont conservés :

- **Mode test** : sous-échantillon de 100 000 lignes pour le développement ;
- **Mode complet** : intégralité du dataset nettoyé pour la validation finale.

Aucune transformation n'est appliquée lors du chargement.

In [2]:
if not INPUT_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset nettoyé introuvable : {INPUT_DATA_PATH}"
    )

if TEST_MODE:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        nrows=NROWS_TEST,
        low_memory=False,
        parse_dates=["registration_date"],
    )

    print(
        f"Mode TEST : {len(df):,} observations chargées."
    )
else:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        low_memory=False,
        parse_dates=["registration_date"],
    )

    print(
        f"Mode COMPLET : {len(df):,} observations chargées."
    )

print(f"Shape : {df.shape}")

Mode TEST : 100,000 observations chargées.
Shape : (100000, 29)


## 2. Contrôle du schéma d'entrée

### Objectif

Avant toute création de features, cette étape vérifie :

- la présence de la variable cible ;
- la présence des principales variables candidates ;
- les types de données ;
- la cohérence du schéma fourni par le notebook 01.

In [3]:
required_columns = {
    "vehicle_record_id",
    "country",
    "manufacturer_name_eu",
    "manufacturer_make",
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
    "mass_running_order_kg",
    "engine_capacity_cm3",
    "engine_power_kw",
    "registration_date",
    TARGET,
}

missing_columns = sorted(
    required_columns - set(df.columns)
)

if missing_columns:
    raise ValueError(
        "Variables attendues absentes : "
        + ", ".join(missing_columns)
    )

print("✅ Schéma d'entrée valide.")

✅ Schéma d'entrée valide.


## 3. Séparation entre cible, identifiants et variables explicatives

### Objectif

Toutes les colonnes présentes dans le dataset nettoyé ne doivent pas
nécessairement être utilisées comme variables prédictives.

Cette étape distingue :

- la variable cible ;
- les colonnes de traçabilité ;
- les variables candidates à la modélisation ;
- les variables dont l'utilisation devra être justifiée ou exclue
  pour éviter une fuite de données.

In [4]:
ID_COLUMNS = [
    "vehicle_record_id",
]

TARGET_COLUMNS = [
    TARGET,
]

candidate_features = [
    column
    for column in df.columns
    if column not in ID_COLUMNS + TARGET_COLUMNS
]

print(f"Nombre de variables candidates : {len(candidate_features)}")

Nombre de variables candidates : 27


## 4. Analyse des variables catégorielles

### 4.1 Identification des variables catégorielles

L'objectif est de mesurer la cardinalité des variables catégorielles avant
de choisir une stratégie d'encodage.

Les encodages ne sont pas encore appliqués à ce stade.

In [5]:
categorical_columns = df[
    candidate_features
].select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [
            df[column].nunique(dropna=False)
            for column in categorical_columns
        ],
        "missing_rate_pct": [
            df[column].isna().mean() * 100
            for column in categorical_columns
        ],
    }
).sort_values(
    "unique_values"
)

display(categorical_summary)

,column,unique_values,missing_rate_pct
12,vehicle_category,1,0.000
11,vehicle_category_type,2,0.000
0,country,3,0.000
14,fuel_mode,6,0.000
13,fuel_type,9,0.000
2,manufacturer_pool,12,4.404
16,emission_standard,32,0.012
15,innovative_technology,73,43.767
4,manufacturer_name_oem,84,0.000
3,manufacturer_name_eu,86,0.000


### 4.2 Classification de la cardinalité

Pour préparer le futur pipeline de preprocessing, les variables
catégorielles sont classées selon leur cardinalité :

- faible cardinalité ;
- cardinalité intermédiaire ;
- forte cardinalité.

Cette classification sert à orienter le choix futur entre One-Hot Encoding,
encodage binaire ou éventuelle suppression / transformation métier.

In [6]:
def classify_cardinality(
    unique_values: int,
) -> str:
    if unique_values <= 15:
        return "low"
    if unique_values <= 100:
        return "medium"
    return "high"


categorical_summary["cardinality_level"] = (
    categorical_summary["unique_values"]
    .apply(classify_cardinality)
)

display(categorical_summary)

,column,unique_values,missing_rate_pct,cardinality_level
12,vehicle_category,1,0.000,low
11,vehicle_category_type,2,0.000,low
0,country,3,0.000,low
14,fuel_mode,6,0.000,low
13,fuel_type,9,0.000,low
2,manufacturer_pool,12,4.404,low
16,emission_standard,32,0.012,medium
15,innovative_technology,73,43.767,medium
4,manufacturer_name_oem,84,0.000,medium
3,manufacturer_name_eu,86,0.000,medium


## 5. Feature Engineering temporel

### 5.1 Extraction des composantes temporelles

La date d'immatriculation n'est pas utilisée directement par les modèles.

Les variables suivantes sont dérivées :

- mois d'immatriculation ;
- composantes cycliques du mois (`sin` / `cos`).

L'année n'est pas encodée cycliquement par défaut : une année n'est pas une
variable périodique. Dans le dataset 2024, elle peut également être constante.

In [7]:
DATE_COLUMN = "registration_date"

df["registration_month"] = (
    df[DATE_COLUMN].dt.month
)

df["registration_month_sin"] = np.sin(
    2
    * np.pi
    * (df["registration_month"] - 1)
    / 12
)

df["registration_month_cos"] = np.cos(
    2
    * np.pi
    * (df["registration_month"] - 1)
    / 12
)

display(
    df[
        [
            DATE_COLUMN,
            "registration_month",
            "registration_month_sin",
            "registration_month_cos",
        ]
    ].head()
)

,registration_date,registration_month,registration_month_sin,registration_month_cos
0,2024-02-21,2,0.5,8.660254e-01
1,2024-12-31,12,-0.5,8.660254e-01
2,2024-10-08,10,-1.0,-1.836970e-16
3,2024-12-30,12,-0.5,8.660254e-01
4,2024-10-16,10,-1.0,-1.836970e-16


### 5.2 Validation des variables temporelles

Cette étape vérifie que les transformations temporelles n'ont introduit
aucune valeur manquante inattendue.

In [8]:
temporal_features = [
    "registration_month",
    "registration_month_sin",
    "registration_month_cos",
]

temporal_missing = (
    df[temporal_features]
    .isna()
    .sum()
)

display(temporal_missing)

if temporal_missing.sum() == 0:
    print("✅ Features temporelles valides.")

registration_month        0
registration_month_sin    0
registration_month_cos    0
dtype: int64

✅ Features temporelles valides.


## 6. Définition de la stratégie d'encodage

### Objectif

Les transformations catégorielles dépendant des données doivent être
apprises uniquement sur le jeu d'entraînement.

Pour cette raison, aucun `fit()` de `OneHotEncoder`, `BinaryEncoder` ou autre
encodeur n'est effectué dans ce notebook avant la séparation train / test.

La stratégie retenue sera utilisée ultérieurement dans un pipeline
`scikit-learn`.

In [9]:
encoding_strategy = categorical_summary.copy()

encoding_strategy["recommended_encoding"] = (
    encoding_strategy["cardinality_level"]
    .map(
        {
            "low": "one_hot",
            "medium": "binary_or_frequency",
            "high": "review_or_high_cardinality_encoding",
        }
    )
)

display(
    encoding_strategy[
        [
            "column",
            "unique_values",
            "cardinality_level",
            "recommended_encoding",
        ]
    ]
)

,column,unique_values,cardinality_level,recommended_encoding
12,vehicle_category,1,low,one_hot
11,vehicle_category_type,2,low,one_hot
0,country,3,low,one_hot
14,fuel_mode,6,low,one_hot
13,fuel_type,9,low,one_hot
2,manufacturer_pool,12,low,one_hot
16,emission_standard,32,medium,binary_or_frequency
15,innovative_technology,73,medium,binary_or_frequency
4,manufacturer_name_oem,84,medium,binary_or_frequency
3,manufacturer_name_eu,86,medium,binary_or_frequency


## 7. Analyse de pertinence des variables candidates

### Objectif

Avant de constituer définitivement le dataset destiné à la modélisation,
cette étape examine les variables candidates afin d'identifier :

- les variables à très forte cardinalité ;
- les identifiants ou quasi-identifiants techniques ;
- les variables susceptibles d'introduire une fuite d'information vis-à-vis
  de la cible `co2_wltp_g_km` ;
- les variables nécessitant une justification métier avant leur utilisation.

Cette analyse permet de distinguer les variables réellement prédictives
des variables qui doivent être exclues ou traitées spécifiquement avant
l'entraînement des modèles.

In [10]:
candidate_analysis = pd.DataFrame({
    "column": [
        column
        for column in df.columns
        if column != TARGET
    ],
    "dtype": [
        str(df[column].dtype)
        for column in df.columns
        if column != TARGET
    ],
    "unique_values": [
        df[column].nunique(dropna=False)
        for column in df.columns
        if column != TARGET
    ],
    "missing_pct": [
        df[column].isna().mean() * 100
        for column in df.columns
        if column != TARGET
    ],
})

candidate_analysis["unique_ratio_pct"] = (
    candidate_analysis["unique_values"]
    / len(df)
    * 100
)

candidate_analysis = candidate_analysis.sort_values(
    "unique_values",
    ascending=False,
)

display(candidate_analysis)

,column,dtype,unique_values,missing_pct,unique_ratio_pct
0,vehicle_record_id,int64,100000,0.000,100.000
9,vehicle_version,object,7068,0.256,7.068
2,vehicle_family_id,object,2990,0.364,2.990
8,vehicle_variant,object,2223,0.232,2.223
11,commercial_name,object,2057,0.003,2.057
15,wltp_test_mass_kg,float64,1958,0.285,1.958
6,type_approval_number,object,1642,0.000,1.642
26,rlfi,object,1546,1.691,1.546
14,mass_running_order_kg,float64,1367,0.000,1.367
27,electric_range_km,float64,543,78.987,0.543


## 8. Analyse de la relation entre certaines variables métier et la cible

### Objectif

Cette étape mesure la relation entre plusieurs variables métier et la cible
`co2_wltp_g_km`.

L'analyse porte en priorité sur :

- `fuel_consumption` ;
- `co2_reduction_wltp_g_km` ;
- `electric_energy_consumption_wh_km` ;
- `electric_range_km`.

Pour chacune de ces variables, les indicateurs suivants sont examinés :

- le nombre d'observations disponibles ;
- le taux de valeurs manquantes ;
- la corrélation linéaire avec la cible.

L'objectif est d'identifier les variables les plus informatives pour la
modélisation et de documenter leur comportement avant la sélection finale
des features.

In [11]:
leakage_candidates = [
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
]

available_leakage_candidates = [
    column
    for column in leakage_candidates
    if column in df.columns
]

leakage_analysis = pd.DataFrame({
    "column": available_leakage_candidates,
    "non_null_count": [
        df[column].notna().sum()
        for column in available_leakage_candidates
    ],
    "missing_pct": [
        df[column].isna().mean() * 100
        for column in available_leakage_candidates
    ],
    "correlation_with_target": [
        df[[column, TARGET]]
        .corr()
        .iloc[0, 1]
        for column in available_leakage_candidates
    ],
})

leakage_analysis["abs_correlation"] = (
    leakage_analysis["correlation_with_target"].abs()
)

leakage_analysis = leakage_analysis.sort_values(
    "abs_correlation",
    ascending=False,
)

display(leakage_analysis)

,column,non_null_count,missing_pct,correlation_with_target,abs_correlation
0,fuel_consumption,85391,14.609,0.967461,0.967461
3,electric_range_km,21013,78.987,-0.727891,0.727891
2,electric_energy_consumption_wh_km,21038,78.962,0.387561,0.387561
1,co2_reduction_wltp_g_km,56233,43.767,0.156082,0.156082


## 9. Sélection des variables pour le dataset de features

### Objectif

À partir de l'analyse de pertinence réalisée précédemment, cette étape définit
explicitement les variables conservées ou exclues du dataset destiné à la
modélisation.

Les exclusions concernent notamment :

- les identifiants purement techniques ;
- la date brute après création des variables temporelles ;
- les variables constantes sans pouvoir prédictif ;
- les variables intermédiaires utilisées uniquement pour construire de nouvelles
  features.

Les variables à forte cardinalité ne sont pas supprimées automatiquement à ce
stade. Leur stratégie de traitement sera définie ultérieurement dans le pipeline
de preprocessing Machine Learning.

De même, les variables présentant un risque potentiel de fuite d'information
sont conservées provisoirement pour analyse métier avant toute décision
définitive.

In [12]:
# ---------------------------------------------------------------------
# Variables exclues du dataset de features
# ---------------------------------------------------------------------

columns_to_exclude = {
    "vehicle_record_id",
    "registration_date",
    "registration_month",
    "vehicle_category",
}

feature_columns = [
    column
    for column in df.columns
    if column not in columns_to_exclude
]

df_features = df[
    feature_columns
].copy()

print(
    f"Dataset de features : "
    f"{len(df_features):,} observations × "
    f"{df_features.shape[1]} variables"
)

print("\nVariables exclues :")
for column in sorted(columns_to_exclude):
    print(f"  - {column}")

Dataset de features : 100,000 observations × 28 variables

Variables exclues :
  - registration_date
  - registration_month
  - vehicle_category
  - vehicle_record_id


## 10. Contrôles qualité du dataset de features

### Objectif

Avant export, cette étape vérifie :

- la présence de la cible ;
- l'absence de doublon de noms de colonnes ;
- la présence des nouvelles variables temporelles ;
- l'absence de l'identifiant technique dans les features.

In [13]:
quality_checks = {
    "Variable cible présente":
        TARGET in df_features.columns,

    "Noms de colonnes uniques":
        df_features.columns.is_unique,

    "Identifiant technique supprimé":
        "vehicle_record_id"
        not in df_features.columns,

    "Date brute supprimée":
        "registration_date"
        not in df_features.columns,

    "Variable constante vehicle_category supprimée":
        "vehicle_category"
        not in df_features.columns,

    "Feature mois sinus présente":
        "registration_month_sin"
        in df_features.columns,

    "Feature mois cosinus présente":
        "registration_month_cos"
        in df_features.columns,
}

for check, result in quality_checks.items():
    status = "✅" if result else "❌"
    print(f"{status} {check}")

if not all(quality_checks.values()):
    raise ValueError(
        "Au moins un contrôle qualité a échoué."
    )

✅ Variable cible présente
✅ Noms de colonnes uniques
✅ Identifiant technique supprimé
✅ Date brute supprimée
✅ Variable constante vehicle_category supprimée
✅ Feature mois sinus présente
✅ Feature mois cosinus présente


## 11. Export du dataset de Feature Engineering

Le dataset produit à cette étape n'est pas encore encodé par des
transformations apprises sur les données.

Il constitue l'entrée du futur pipeline de séparation train / test et de
préprocessing Machine Learning.

In [14]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / (
        "data_2024_features_test.csv"
        if TEST_MODE
        else "data_2024_features.csv"
    )
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("✅ Dataset de features exporté.")
print(f"Fichier      : {OUTPUT_PATH}")
print(f"Observations : {len(df_features):,}")
print(f"Variables    : {df_features.shape[1]}")

✅ Dataset de features exporté.
Fichier      : /home/jmbandong/projects/ml-projects/vehicle-emissions-prediction-mlops/data/interim/data_2024_features_test.csv
Observations : 100,000
Variables    : 28


## 12. Vérification finale du dataset de features

### Objectif

Cette étape réalise un contrôle final du dataset de features généré avant de poursuivre vers la séparation des jeux d'entraînement et de test.

Les vérifications portent sur :

- l'aperçu des premières observations ;
- la structure générale du DataFrame ;
- le nombre final d'observations et de variables ;
- la présence de la variable cible ;
- la cohérence du schéma obtenu après Feature Engineering.

Cette étape ne modifie pas les données. Elle constitue uniquement un contrôle de validation du dataset produit.

In [15]:
display(df_features.head())

print("\nInformations générales du dataset de features :")
df_features.info()

print("\nRésumé final :")
print(f"Shape finale          : {df_features.shape}")
print(
    f"Variable cible présente : "
    f"{TARGET in df_features.columns}"
)

,country,vehicle_family_id,manufacturer_pool,manufacturer_name_eu,manufacturer_name_oem,type_approval_number,vehicle_type,vehicle_variant,vehicle_version,manufacturer_make,...,engine_power_kw,electric_energy_consumption_wh_km,innovative_technology,co2_reduction_wltp_g_km,fuel_consumption,emission_standard,rlfi,electric_range_km,registration_month_sin,registration_month_cos
0,FR,IP-BX72_2021_00005-WF0-1,FORD,FORD WERKE GMBH,FORD-WERKE GMBH,e9*2007/46*3165*15,J2K,B7JG12X,5CFETNA5PAX,FORD,...,91.0,NaN,E9 32 37,2.00,5.4,Euro 6 AP,RL-B479_2019_00001-WF0-1,NaN,0.5,8.660254e-01
1,FR,IP-MQB27SZ_A1_1021-WVW-1,VOLKSWAGEN,VOLKSWAGEN,VOLKSWAGEN AG,e13*2018/858*00140*06,CS,ACDLAA,FD7FD7CW0094BIA1BI0,VOLKSWAGEN,...,81.0,NaN,E13 29,1.17,6.0,Euro 6 AP,RL-DQ200_7F_17_012-WVW-1,NaN,-0.5,8.660254e-01
2,FR,IP-HNA1M2PDB1A_000-VF1,RENAULT-NISSAN-MITSUBISHI,RENAULT,RENAULT SAS,e9*2018/858*30002*13,RHN,DH2,R25W3620000B,RENAULT,...,96.0,NaN,E9 37,0.68,5.0,Euro 6 EA,RL-HNCDB1A_450_000-VF1,NaN,-1.0,-1.836970e-16
3,FR,IP-6_004593-TSM-1,VOLVO CARS POLESTAR SUZUKI,MAGYAR SUZUKI,MAGYAR SUZUKI CORPORATION LTD,e4*2007/46*0779*18,JY,AH2S,AGS,SUZUKI,...,75.0,NaN,E6 37,0.81,5.1,Euro 6 EA,RL-6_LY8004-TSM-1,NaN,-0.5,8.660254e-01
4,FR,IP-FKA1JAE044A_000-VF1,RENAULT-NISSAN-MITSUBISHI,RENAULT,RENAULT SAS,e2*2018/858*00001*13,RFK,RKJ,JA0AT00410B0,RENAULT,...,90.0,194.0,NaN,NaN,NaN,AX,RL-FKARA0A_245_000-VF1,280.0,-1.0,-1.836970e-16



Informations générales du dataset de features :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   country                            100000 non-null  object 
 1   vehicle_family_id                  99636 non-null   object 
 2   manufacturer_pool                  95596 non-null   object 
 3   manufacturer_name_eu               100000 non-null  object 
 4   manufacturer_name_oem              100000 non-null  object 
 5   type_approval_number               100000 non-null  object 
 6   vehicle_type                       99984 non-null   object 
 7   vehicle_variant                    99768 non-null   object 
 8   vehicle_version                    99744 non-null   object 
 9   manufacturer_make                  99996 non-null   object 
 10  commercial_name                    99997 non-null   obje